# Lab 10A: One Loop, Many Techniques — Looking Inside a Frozen Network

**Before you start: Runtime → Change runtime type → GPU.**

This week's lab consists of **three notebooks**:

| Notebook | Question |
|---|---|
| **10A** One loop, many techniques | What can a frozen network tell us when we optimise the *input* instead of the weights? |
| **10B** Right for the right reasons? | Our model is 100% accurate on the validation set. Is it looking at the right thing, and can we trust the tools that tell us? |
| **10C** *(optional)* Gram matrices, texture and style | What does "style" mean to a network? |

**10B is the most important notebook.** If you are short of time, skip 10C, not 10B. Afterwards,
complete this week's pages in your iteration log: you will use the tools from 10B on **your own model**.

**Theory ↔ code.** As in Lab 9, boxes marked **🔗 Theory ↔ code** show how the equations and
procedures from the lecture map to lines of PyTorch.

**Cells marked 🛠️ TASK contain gaps (`???`) that you must fill in.** Most come with a test cell; don't
move on until the tests pass. Write down your ✍️ **Prediction** before running an experiment.

### This notebook (10A)

The lecture presented many techniques: saliency maps, inverting a layer, filter visualisation,
class visualisation, adversarial examples, style transfer. They look very different, but almost all
of them do the **same thing**:

> Freeze the network. Define a scalar loss on some activation or output. Differentiate the loss with
> respect to the **input pixels**, and change the pixels.

In this notebook, the loop that does this is given. For each technique, your job is to write its
**loss**: one short line that says what question we ask the network. Each line corresponds to a row of
the lecture's summary table (Section 10 of the notes).

| Part | You will… |
|---|---|
| 1. Looking inside | capture the activations of every layer of VGG16 for an image, and see how they change with depth |
| 2. One loop, many losses | write the loss for gradient saliency, layer inversion, filter visualisation, class visualisation and adversarial examples |
| 3. Exam-style questions | check your understanding |

## Setup

The setup cell defines a few **helper functions**, so that the code in the tasks reads like the
lecture's equations. You don't need to understand how they work inside, only what they return:

| Helper | Returns | Lecture notation |
|---|---|---|
| `class_score(x, c)` | the logit of class `c` for image `x` (a single number) | $y_c$ |
| `activation(x, layer)` | the feature maps of `vgg.features[layer]` for image `x`, shape (C, H, W) | $\mathbf{a}^l(\mathbf{x})$ |
| `filter_activation(x, layer, k)` | the mean of feature map `k` of that layer (a single number) | mean of $\mathbf{a}^l_k(\mathbf{x})$ |
| `gradient_of(f, x)` | the gradient of the number `f(x)` with respect to every pixel of `x` | $\partial f / \partial \mathbf{x}$ |

All images are tensors of shape (1, 3, H, W) with pixel values in $[0, 1]$. The ImageNet
normalisation that VGG16 needs happens inside the helpers.

In [ ]:
import copy
import math
import random
import time

import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchvision
import torchvision.transforms as T
import torchvision.transforms.functional as TF

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

# ---- Images: the example images from the lecture, stored in the course repository ----
import os, urllib.request
from PIL import Image

REPO = 'https://raw.githubusercontent.com/klaverhenrik/Deep-Learning-for-Visual-Recognition-2026/main/data/'
os.makedirs('images', exist_ok=True)


def get_image(name):
    path = os.path.join('images', name)
    if not os.path.exists(path):
        urllib.request.urlretrieve(REPO + name, path)
    return Image.open(path).convert('RGB')


IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)
normalise = lambda x: (x - IMAGENET_MEAN) / IMAGENET_STD        # the preprocessing the network was trained with


def load_image(name, size=224):
    tfm = T.Compose([T.Resize(size), T.CenterCrop(size), T.ToTensor()])
    return tfm(get_image(name)).unsqueeze(0).to(device)          # (1, 3, size, size), pixels in [0, 1]


def show(x):
    # (1, 3, H, W) in [0, 1]  ->  (H, W, 3), for plt.imshow
    return x[0].detach().clamp(0, 1).permute(1, 2, 0).cpu()


def stretch(x):
    # For optimised images with very low contrast: stretch to the full range [0, 1] for display
    x = x[0].detach().cpu()
    return ((x - x.min()) / (x.max() - x.min() + 1e-8)).permute(1, 2, 0)


# ---- VGG16, pretrained on ImageNet (the download is about 0.5 GB, so it takes a little while) ----
weights = torchvision.models.VGG16_Weights.IMAGENET1K_V1
categories = weights.meta['categories']
vgg = torchvision.models.vgg16(weights=weights).to(device).eval()
for p in vgg.parameters():
    p.requires_grad_(False)                     # FREEZE the network: we never change its weights in this lab
for m in vgg.modules():
    if isinstance(m, nn.ReLU):
        m.inplace = False


# ---- Helper functions ----
def class_scores(x):
    return vgg(normalise(x))                    # logits, shape (N, 1000)


def class_score(x, c):
    return class_scores(x)[0, c]                # the logit y_c of class c for one image


def activation(x, layer):
    return vgg.features[:layer + 1](normalise(x))[0]      # feature maps of vgg.features[layer], shape (C, H, W)


def filter_activation(x, layer, k):
    return activation(x, layer)[k].mean()       # mean of feature map k


def gradient_of(f, x):
    x = x.clone().requires_grad_(True)
    f(x).backward()
    return x.grad.detach()                      # same shape as x


cat = load_image('Cat03.jpg')
with torch.no_grad():
    probs = F.softmax(class_scores(cat), dim=1)[0]
top = probs.topk(3)
print('VGG16 says:', [(categories[i], round(p.item(), 2)) for p, i in zip(top.values, top.indices)])
plt.figure(figsize=(3, 3)); plt.imshow(show(cat)); plt.axis('off'); plt.show()

---
# Part 1 — Looking inside

To see what happens *inside* a network, we need the outputs of its intermediate layers. PyTorch
lets us attach a **forward hook** to any layer: a function that is called with the layer's output
every time the layer runs. The function `get_activations` below uses hooks to collect the output of
**every** layer of `vgg.features` for one image. Read it; Question 1.1 asks what it does.

### 🔗 Theory ↔ code

| Lecture | PyTorch |
|---|---|
| VGG16's layers `conv1_1`, `relu1_1`, …, `conv5_3`, `relu5_3` | `vgg.features[0]`, `vgg.features[1]`, …, `vgg.features[28]`, `vgg.features[29]` |
| "plot the feature maps of layer $l$" | `acts[l][0, k]` is feature map $k$ of layer $l$ |
| the first-layer filters, shown as colour images | `vgg.features[0].weight`, shape (64, 3, 3, 3): 64 filters of size 3×3 with 3 colour channels |

In [ ]:
def get_activations(x):
    acts, handles = {}, []

    def make_hook(i):
        def hook(module, inputs, output):
            acts[i] = output.detach()           # store the output of layer i
        return hook

    for i, layer in enumerate(vgg.features):
        handles.append(layer.register_forward_hook(make_hook(i)))
    with torch.no_grad():
        vgg(normalise(x))                       # ONE forward pass fills `acts`
    for h in handles:
        h.remove()                              # remove the hooks again
    return acts


acts = get_activations(cat)
print('Captured', len(acts), 'layers. For example, relu5_3:', tuple(acts[29].shape))

### ✍️ TASK 1: Predict, then measure

For each ReLU layer, we measure the fraction of activations that are **exactly zero**, and the
fraction of feature maps that are **completely** zero (all-black) for this image. We also count the
exact zeros at the output of the convolution `conv5_3`, just **before** its ReLU.

Before you run the cell, write down your predictions:
* How will the fraction of zeros change from `relu1_1` to `relu5_3`?
* How many exact zeros do you expect at the output of `conv5_3`, compared with `relu5_3`?

In [ ]:
relu_names = {1: 'relu1_1', 3: 'relu1_2', 6: 'relu2_1', 8: 'relu2_2', 11: 'relu3_1', 13: 'relu3_2', 15: 'relu3_3',
              18: 'relu4_1', 20: 'relu4_2', 22: 'relu4_3', 25: 'relu5_1', 27: 'relu5_2', 29: 'relu5_3'}
print(f'{"layer":10s}{"shape":>18s}{"zeros":>9s}{"all-zero maps":>15s}')
for i, name in relu_names.items():
    a = acts[i]
    zeros = (a == 0).float().mean().item()
    dead_maps = (a.flatten(2).amax(2) == 0).float().mean().item()
    print(f'{name:10s}{str(tuple(a.shape[1:])):>18s}{zeros:9.0%}{dead_maps:15.0%}')
print(f'\nconv5_3 (before the ReLU): {(acts[28] == 0).float().mean().item():.1%} zeros')
print(f'relu5_3 (after the ReLU):  {(acts[29] == 0).float().mean().item():.1%} zeros')

fig, axs = plt.subplots(3, 8, figsize=(16, 6.5))
w = vgg.features[0].weight.detach().cpu()
a1, a5 = acts[1][0].cpu(), acts[29][0].cpu()
top8 = a5.flatten(1).amax(1).argsort(descending=True)[:8]
for k in range(8):
    f = w[k]; axs[0, k].imshow(((f - f.min()) / (f.max() - f.min())).permute(1, 2, 0), interpolation='nearest')
    axs[1, k].imshow(a1[k], cmap='gray')
    axs[2, k].imshow(a5[top8[k]], cmap='gray')
for ax in axs.flat:
    ax.axis('off')
axs[0, 0].set_title('conv1_1 filters (3×3, RGB)', loc='left')
axs[1, 0].set_title('relu1_1 feature maps', loc='left')
axs[2, 0].set_title('relu5_3: the 8 most active of 512 feature maps', loc='left')
plt.tight_layout(); plt.show()

### Questions 1

1. Read `get_activations`. What does a forward hook do, and why do we get the outputs of *all* layers from
   a single forward pass?
2. Describe how the fraction of zeros and the fraction of all-zero feature maps change from `relu1_1`
   to `relu5_3`. Compare with the three rows of images.
3. In `relu5_3`, many feature maps are completely black for this cat. Is something wrong with the
   network? What would make you worried?
4. Exercise 2 in the lecture notes hooks the **convolution** layers `features[0]` and `features[28]`, and
   counts exact zeros as "dead". Why does that not work? Which layers should it hook?
5. What do the first-layer filters look like? Why can we show the first-layer filters as colour images,
   but not the filters of deeper layers?

---
# Part 2 — One loop, many losses

### 🔗 Theory ↔ code: training turned around

| | Training (Labs 6–9) | This lab |
|---|---|---|
| What is optimised | the weights $\mathbf{W}$ | the **input image** $\mathbf{x}$ |
| What is fixed | the training images | the network: `requires_grad_(False)` on every weight |
| The optimiser | `Adam(model.parameters())` | ? (Task 2a) |
| The loss | how wrong the prediction is | **the question we ask** (Tasks 2b–2f) |
| Extra step | — | a **projection** after each step that keeps $\mathbf{x}$ "valid" (e.g. pixels in $[0, 1]$) |

Below is the loop. It is the training loop you know from Labs 6–9, with two differences: the "parameter"
is the image `x`, and after each step a `project` function can modify `x` (e.g. clip it to $[0, 1]$).

### 🛠️ TASK 2a: What does the optimiser update?

Fill in the one missing argument: the list of tensors the optimiser should change.

In [ ]:
def optimise_input(loss_fn, x_init, steps, lr, project=None):
    x = x_init.clone().requires_grad_(True)        # the image is now the "parameter"
    opt = torch.optim.Adam(???, lr=lr)             # what should the optimiser change?
    for step in range(steps):
        opt.zero_grad()
        loss = loss_fn(x)                          # the question we ask the network
        loss.backward()                            # d loss / d x: the gradient with respect to every pixel
        opt.step()                                 # change the pixels
        if project is not None:
            with torch.no_grad():
                project(x, step)                   # keep the image valid
    return x.detach()


def keep_valid(x, step):
    x.clamp_(0, 1)                                 # every pixel stays between 0 and 1

In [ ]:
# Test for Task 2a
_x = optimise_input(lambda x: ((x - 3) ** 2).mean(), torch.zeros(1, 3, 4, 4), steps=500, lr=0.1)
assert torch.allclose(_x, torch.full_like(_x, 3.0), atol=0.05), 'x should move to the minimum of the loss (3)'
_x = optimise_input(lambda x: -class_score(x, 0), cat, steps=1, lr=0.01)
assert not torch.equal(_x, cat), 'the image should change'
assert all(p.grad is None for p in vgg.parameters()), "the network's weights must not get gradients"
print('✅ Task 2a passed')


def noise_image():
    # The starting point for the optimisations below: an almost uniform grey image with a little noise
    return (IMAGENET_MEAN + 0.002 * torch.randn(1, 3, 224, 224, device=device)).clamp(0, 1)

### 2.1 Gradient saliency: the loop with zero steps

The simplest technique doesn't change the image at all: we compute the gradient of the class score
$y_c$ (the **logit**, before the softmax) with respect to the pixels once, and show its size:

$$\text{saliency}_{ij} = \max_\text{colour} \left| \frac{\partial y_c}{\partial x_{ij}} \right|.$$

### 🛠️ TASK 2b: Which number do we differentiate?

| Lecture | Code |
|---|---|
| $\dfrac{\partial y_c}{\partial \mathbf{x}}$ | `gradient_of(f, cat)` with `f(x)` $= y_c$ = `class_score(x, cat_idx)` |

`gradient_of(f, cat)` differentiates the number `f(x)`. Fill in `f`: the score of the predicted class
`cat_idx`.

In [ ]:
cat_idx = probs.argmax().item()
f = lambda x: ???                                  # the logit of class cat_idx
grad = gradient_of(f, cat)                         # shape (1, 3, 224, 224)
saliency = grad.abs().amax(dim=1)[0]               # |gradient|, maximum over the colour channels: (224, 224)

fig, axs = plt.subplots(1, 2, figsize=(8, 4))
axs[0].imshow(show(cat)); axs[1].imshow(saliency.cpu(), cmap='gray', vmax=saliency.quantile(0.99).item())
axs[1].set_title(f'gradient saliency: {categories[cat_idx]}')
for ax in axs: ax.axis('off')
plt.show()

In [ ]:
# Test for Task 2b
assert saliency.shape == (224, 224) and saliency.max() > 0
print('✅ Task 2b passed')

### 2.2 Inverting a layer: what does a layer remember?

Given the activations $\mathbf{a}^l(\mathbf{x})$ of the cat at layer $l$, find an image $\mathbf{y}$ with the same
activations, starting from noise (lecture, Section 5.1). The loss is

$$\mathcal{L}(\mathbf{y}) = \frac{\|\mathbf{a}^l(\mathbf{y}) - \mathbf{a}^l(\mathbf{x})\|^2}{\|\mathbf{a}^l(\mathbf{x})\|^2} \; + \; \lambda_\text{TV} \cdot \text{TV}(\mathbf{y}).$$

The first term is the squared distance from the lecture ($\|\mathbf{a}^l(\mathbf{y}) - \mathbf{a}^l(\mathbf{x})\|_F^2$), divided by
$\|\mathbf{a}^l(\mathbf{x})\|^2$ so that the same learning rate works for every layer. The second term is optional:
the **total variation** $\text{TV}(\mathbf{y})$ is the average absolute difference between neighbouring
pixels, which penalises high-frequency noise. It is a regulariser for the *image*, just as weight decay
is a regulariser for the weights: many images have the same activations, and the TV term prefers the
smooth ones.

### 🛠️ TASK 2c: The inversion loss

| Lecture | Code |
|---|---|
| $\mathbf{a}^l(\mathbf{y})$ | `activation(y, layer)` |
| $\mathbf{a}^l(\mathbf{x})$ (the cat's activations, computed once) | `target` |
| $\dfrac{\|\mathbf{a} - \mathbf{b}\|^2}{\|\mathbf{b}\|^2}$ | `distance(a, b)` |
| $\lambda_\text{TV} \cdot \text{TV}(\mathbf{y})$ | `tv_weight * total_variation(y)` |

✍️ **Prediction:** what will the reconstructions from `relu1_2` and from `relu5_3` look like?

In [ ]:
def distance(a, b):
    return ((a - b) ** 2).sum() / (b ** 2).sum()


def total_variation(x):
    return (x[..., 1:, :] - x[..., :-1, :]).abs().mean() + (x[..., :, 1:] - x[..., :, :-1]).abs().mean()


def invert(layer, tv_weight, steps=200):
    with torch.no_grad():
        target = activation(cat, layer)            # a^l(x): the cat's activations at this layer
    loss_fn = lambda y: ???                        # the loss L(y) above, using distance, activation and total_variation
    return optimise_input(loss_fn, noise_image(), steps=steps, lr=0.01, project=keep_valid)


layers = {3: 'relu1_2', 8: 'relu2_2', 15: 'relu3_3', 22: 'relu4_3', 29: 'relu5_3'}
t0 = time.time()
fig, axs = plt.subplots(2, 6, figsize=(18, 6.5))
for r, tv in enumerate([0.0, 0.5]):
    axs[r, 0].imshow(show(cat)); axs[r, 0].set_title('original')
    for j, (layer, name) in enumerate(layers.items()):
        axs[r, j + 1].imshow(show(invert(layer, tv))); axs[r, j + 1].set_title(f'from {name}, TV weight {tv}')
for ax in axs.flat: ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### Questions 2.2

1. Describe how the reconstructions change from `relu1_2` to `relu5_3`. What is kept, and what is lost?
2. Every reconstruction has (almost) the same activations as the original at that layer. What does that
   tell you about the network: is it invertible?
3. What did the TV term change? Try a stronger TV weight (change `0.5` to `2.0` and rerun). Is the
   TV-regularised image a "more correct" reconstruction?

### 2.3 Filter visualisation: what does one filter want to see?

Start from noise and change the image so that the **mean activation of one filter** becomes as large
as possible. After each step we keep the pixels valid, and every 10 steps we blur the image slightly,
to suppress high-frequency noise (the same role as the TV term above).

We want to **maximise** the mean activation of filter $k$ in layer $l$, but the optimiser **minimises**
the loss, so the loss is its negative (gradient *ascent* on the activation):

$$\mathcal{L}(\mathbf{x}) = -\frac{1}{HW}\sum_{i,j} a^l_{k,ij}(\mathbf{x}).$$

### 🛠️ TASK 2d: The filter-visualisation loss

| Lecture | Code |
|---|---|
| $\frac{1}{HW}\sum_{i,j} a^l_{k,ij}(\mathbf{x})$: the mean of feature map $k$ of layer $l$ | `filter_activation(x, layer, k)` |

In [ ]:
def visualise_filter(layer, k, steps=150):
    loss_fn = lambda x: ???                        # make the mean activation of filter k in this layer as LARGE as possible

    def project(x, step):
        keep_valid(x, step)
        if step % 10 == 9:
            x.copy_(TF.gaussian_blur(x, kernel_size=3, sigma=0.5))

    return optimise_input(loss_fn, noise_image(), steps=steps, lr=0.02, project=project)


t0 = time.time()
fig, axs = plt.subplots(3, 4, figsize=(12, 9.5))
for r, (layer, name) in enumerate({0: 'conv1_1', 14: 'conv3_3', 28: 'conv5_3'}.items()):
    for c, k in enumerate([0, 1, 2, 3]):
        axs[r, c].imshow(stretch(visualise_filter(layer, k))); axs[r, c].set_title(f'{name}, filter {k}'); axs[r, c].axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### 2.4 Class visualisation: what does the network think a "dumbbell" looks like?

The same idea with the **class score** as the objective (class-based reconstruction): find an image
that maximises the logit $y_c$ of one class. Without help, this produces high-frequency noise that the
network happens to like, so we add three regularisers:

* **jitter**: at every step, the image is shifted by a few random pixels (`jitter(x)`) before it goes
  into the network. A pattern only survives if it raises the score *wherever* it is, which suppresses
  pixel-exact noise;
* a small penalty on how far the image is from plain grey, `size(x)` $= \|\mathbf{x} - \text{grey}\|^2$,
  weighted by $\lambda$ (like weight decay, but for pixels);
* a slight blur every 10 steps (as for the filters).

$$\mathcal{L}(\mathbf{x}) = -\,y_c(\tilde{\mathbf{x}}) + \lambda\,\|\mathbf{x} - \bar{\mathbf{x}}\|^2,$$

where $\tilde{\mathbf{x}}$ is the jittered (shifted) image and $\bar{\mathbf{x}}$ is plain grey.

The lecture showed that for "dumbbell", such images can contain arms. Let's see for ourselves.

### 🛠️ TASK 2e: The class-visualisation loss

| Lecture | Code |
|---|---|
| $\tilde{\mathbf{x}}$ (the shifted image) | `jitter(x)` |
| $y_c(\tilde{\mathbf{x}})$ | `class_score(jitter(x), c)` |
| $\lambda\,\|\mathbf{x} - \bar{\mathbf{x}}\|^2$ | `lam * size(x)` |

In [ ]:
def jitter(x):
    dx, dy = random.randint(-8, 8), random.randint(-8, 8)
    return torch.roll(x, shifts=(dx, dy), dims=(2, 3))       # shift the image by up to 8 pixels


def size(x):
    return ((x - IMAGENET_MEAN) ** 2).sum()                  # how far the image is from plain grey


def visualise_class(c, steps=200, lam=2e-3):
    loss_fn = lambda x: ???                        # the loss L(x) above, with lam as the weight of size(x)

    def project(x, step):
        keep_valid(x, step)
        if step % 10 == 9:
            x.copy_(TF.gaussian_blur(x, kernel_size=3, sigma=0.5))

    return optimise_input(loss_fn, noise_image(), steps=steps, lr=0.02, project=project)


t0 = time.time()
classes = [543, 1, 949]                                  # dumbbell, goldfish, strawberry
fig, axs = plt.subplots(1, 3, figsize=(15, 5))
for ax, c in zip(axs, classes):
    x = visualise_class(c)
    with torch.no_grad():
        p = F.softmax(class_scores(x), dim=1)[0, c].item()
    ax.imshow(show(x)); ax.set_title(f'{categories[c]}  (VGG16: p = {p:.2f})'); ax.axis('off')
plt.tight_layout(); plt.show()
print(f'Time: {time.time() - t0:.0f} s')

### Questions 2.3–2.4

1. Every `conv1_1` image is a **single uniform colour**. Why? (Hint: `conv1_1` is one linear 3×3 filter, and
   the loss is its *mean* output over all positions. What does that mean depend on?) Is this what the
   `conv1_1` filter "detects"?
2. How do the preferred patterns change from `conv3_3` to `conv5_3`?
3. Can you recognise the classes in the class visualisations? Do you see parts that are not the object
   itself (like the arms for "dumbbell")? What would that tell you about the training data?
4. VGG16 is (almost) certain that these images show the class. Would a human agree? What does that tell
   you about the difference between "an image the network is confident about" and "a typical image of
   the class"?
5. Try the class visualisation without jitter (replace `jitter(x)` by `x` in your loss) for one class. What
   changes?

### 2.5 Adversarial examples: the same idea, used against the network

An **adversarial example** is an image that looks unchanged to a human but changes the network's
prediction. The Fast Gradient Sign Method (FGSM) takes a **single** step: compute the gradient of the
loss for the correct (here: the predicted) class, and move every pixel by $\pm\varepsilon$ in the direction
that *increases* the loss:

$$\mathbf{x}_\text{adv} = \mathbf{x} + \varepsilon \cdot \text{sign}\left(\nabla_\mathbf{x} \mathcal{L}(\mathbf{x}, y)\right).$$

Pixels lie in $[0, 1]$, so $\varepsilon = 0.01$ means "change each pixel by at most 1% of its range",
invisible to a human.

We measure **how often the prediction changes** for a set of images, as a function of $\varepsilon$, and whether
the same adversarial images also fool a **different** network (ResNet-18), which was never used to
compute them (transfer).

### 🛠️ TASK 2f: FGSM

| Lecture | Code |
|---|---|
| $\nabla_\mathbf{x} \mathcal{L}(\mathbf{x}, y)$ | `grad` (already computed) |
| $\text{sign}(\cdot)$ | `.sign()` |
| $\varepsilon$ | `eps` |

Write the FGSM equation.

In [ ]:
names = ['Cat03.jpg', 'Cute_dog.jpg', 'Collage_of_Nine_Dogs.jpg', 'LeafySeaDragon.jpg', 'Good_Food_Display_-_NCI_Visuals_Online.jpg',
         'Kitano_Street_Kobe01s5s4110.jpg', 'Inside_the_shopping_mall_CC_Mart.jpg', 'paris.jpg', 'cat_and_dog.jpg', 'starry_night.jpg']
X = torch.cat([load_image(n) for n in names])
resnet = torchvision.models.resnet18(weights=torchvision.models.ResNet18_Weights.IMAGENET1K_V1).to(device).eval()
for p in resnet.parameters():
    p.requires_grad_(False)
resnet_scores = lambda x: resnet(normalise(x))
with torch.no_grad():
    pred_vgg, pred_resnet = class_scores(X).argmax(1), resnet_scores(X).argmax(1)


def fgsm(x, y, eps):
    grad = gradient_of(lambda x: F.cross_entropy(class_scores(x), y), x)     # gradient of the loss L(x, y)
    x_adv = ???                                    # the FGSM equation
    return x_adv.clamp(0, 1)                       # keep the pixels valid


print(f'{"eps":>6s} {"VGG16 changed":>14s} {"ResNet-18 changed (transfer)":>30s}')
for eps in [0.0, 0.001, 0.003, 0.01, 0.03]:
    x_adv = fgsm(X, pred_vgg, eps)
    with torch.no_grad():
        changed_vgg = (class_scores(x_adv).argmax(1) != pred_vgg).float().mean().item()
        changed_resnet = (resnet_scores(x_adv).argmax(1) != pred_resnet).float().mean().item()
    print(f'{eps:6.3f} {changed_vgg:14.0%} {changed_resnet:30.0%}')

### A targeted attack with the same loop

FGSM only asks for *any* wrong prediction. With `optimise_input` we can ask for a **specific** one: make
VGG16 call the cat a "goldfish", while no pixel changes by more than $\varepsilon = 0.01$:

$$\min_{\mathbf{x}} \; -y_\text{goldfish}(\mathbf{x}) \quad \text{subject to} \quad |x_{ij} - x^\text{cat}_{ij}| \le \varepsilon \text{ for every pixel.}$$

The constraint is just another projection (`stay_close`); the loss is the first part.

### 🛠️ TASK 2g: The targeted-attack loss

| Lecture | Code |
|---|---|
| $y_\text{goldfish}(\mathbf{x})$ | `class_score(x, goldfish)` |
| the constraint $|x_{ij} - x^\text{cat}_{ij}| \le \varepsilon$ | `stay_close` (given) |

✍️ **Prediction:** will you be able to see the difference between the two images?

In [ ]:
eps = 0.01
goldfish = 1


def stay_close(x, step):
    x.copy_((cat + (x - cat).clamp(-eps, eps)).clamp(0, 1))     # |change| <= eps for every pixel, and pixels in [0, 1]


loss_fn = lambda x: ???                            # make the goldfish score as LARGE as possible
x_target = optimise_input(loss_fn, cat, steps=50, lr=0.002, project=stay_close)

with torch.no_grad():
    p = F.softmax(class_scores(x_target), dim=1)[0]
diff = (show(x_target) - show(cat)).abs()
fig, axs = plt.subplots(1, 3, figsize=(13, 4.5))
axs[0].imshow(show(cat)); axs[0].set_title(f'original: {categories[cat_idx]} ({probs[cat_idx]:.2f})')
axs[1].imshow(show(x_target)); axs[1].set_title(f'attacked: {categories[p.argmax()]} ({p.max():.2f})')
axs[2].imshow(diff / diff.max()); axs[2].set_title(f'difference, amplified (max {diff.max():.3f})')
for ax in axs: ax.axis('off')
plt.show()

### Questions 2.5

1. How large must $\varepsilon$ be to change most of VGG16's predictions? Would you see a change of that size
   in an image?
2. Did the adversarial images also fool ResNet-18? What does it mean that adversarial examples
   **transfer** between networks?
3. Compare your targeted-attack loss with your class-visualisation loss. What is the one difference that
   turns a visualisation into an attack?
4. **Project perspective:** why should anyone who deploys a vision model care about this, even if nobody
   is trying to attack it?

### 2.6 Summary: one loop, many questions

You have now written one row of the lecture's summary table for each technique. Fill in the table from your
own code (Section 10 of the lecture notes is the reference).

| Technique | Loss (`loss_fn`) | Steps | Projection | What it reveals |
|---|---|---|---|---|
| Gradient saliency | | | | |
| Layer inversion | | | | |
| Filter visualisation | | | | |
| Class visualisation | | | | |
| FGSM | | | | |
| Targeted attack | | | | |

---
# Part 3 — Exam-style questions

**Q1.** In all the techniques of this notebook, what is optimised and what is kept fixed?

* (a) The weights are optimised; the input image is fixed.
* (b) The input image is optimised; the weights are fixed.
* (c) Both the weights and the image are optimised.
* (d) Nothing is optimised; we only run forward passes.

**Q2.** Why is the loss in filter visualisation the **negative** mean activation of the filter?

* (a) Because activations after a ReLU are negative.
* (b) Because the optimiser minimises the loss, and we want to *maximise* the activation.
* (c) To make the image darker.
* (d) Because the gradient of the mean is negative.

**Q3.** A layer inversion from `relu5_3` gives an image that looks quite different from the original, yet
has almost the same `relu5_3` activations. What does this show?

* (a) The optimisation failed.
* (b) The network is invertible.
* (c) Many different images map to the same deep activations: deep layers discard information such as exact colours and details.
* (d) `relu5_3` is a dead layer.

**Q4.** In FGSM, $\mathbf{x}_\text{adv} = \mathbf{x} + \varepsilon\cdot\text{sign}(\nabla_\mathbf{x}\mathcal{L})$. What does $\varepsilon$ control?

* (a) The learning rate of the network.
* (b) The maximum change of each pixel.
* (c) The number of attack steps.
* (d) Which class the attack targets.

---
**Next: notebook 10B**, where we use these tools to find out whether a model is right for the right
reasons.